In [0]:
import pyspark
orders_df=spark.read.option('header','true').csv("/Volumes/workspace/ds_27/batch_27/orders.csv")
custoemrs_df=spark.read.option('header','true').csv("/Volumes/workspace/ds_27/batch_27/customers.csv")
product_df=spark.read.option('header','true').csv("/Volumes/workspace/ds_27/batch_27/products_level2.csv")
orders_df.show()
custoemrs_df.show()
product_df.show()
orders_df.printSchema()
custoemrs_df.printSchema()
product_df.printSchema()

In [0]:
#valiadte null values from theree tables

orders_df.filter(orders_df.customer_id.isNull()).count()    
custoemrs_df.filter(custoemrs_df.customer_id.isNull()).count()
product_df.filter(product_df.product_id.isNull()).count()

In [0]:
orders_df.filter("quantity <= 0").show()


In [0]:
final_df=orders_df.join(product_df,"product_id").join(custoemrs_df,"customer_id")

In [0]:
final_df.createOrReplaceTempView("sales_view")

In [0]:
%sql
SELECT
customer_name,
SUM(sales_amount) revenue
FROM sales_view
GROUP BY customer_name
ORDER BY revenue DESC

In [0]:
%sql
select city,
sum(sales_amount) as revenue
from sales_view
group by city
order by revenue desc

In [0]:
%sql
select 
product_name,
sum(quantity) as qty
from sales_view
group by product_name
order by qty desc



In [0]:
customer_revenue_df = spark.sql("""
SELECT
  city,
  customer_name,
  revenue
FROM (
  SELECT
    city,
    customer_name,
    SUM(sales_amount) revenue,
    ROW_NUMBER() OVER(
      PARTITION BY city
      ORDER BY SUM(sales_amount) DESC
    ) rn
  FROM sales_view
  GROUP BY city, customer_name
)
WHERE rn = 1
""")

display(customer_revenue_df)

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.gold")

customer_revenue_df.write \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.customer_revenue")